# 🤖 Analisis Sentimen: AI Menggantikan Manusia

**Sumber data:** Komentar TikTok (Neysa 1 + Neysa 2 + Rizky)

**Pipeline:**
1. Merge & Preprocessing teks
2. Language detection + Auto-labeling via Transformer
3. Training SVM & Evaluasi
4. Visualisasi hasil

---

## ⚙️ Install Dependencies
*(Jalankan sekali, skip jika sudah terinstall)*

In [ ]:
# !pip install pandas langdetect transformers torch scikit-learn matplotlib seaborn wordcloud sentencepiece protobuf

---
## 📥 Step 1 — Merge Data & Preprocessing Teks

In [ ]:
import re
import os
import pandas as pd

os.makedirs("output", exist_ok=True)

In [ ]:
# --- Load 3 sumber data ---
neysa1 = pd.read_csv("Neysa_Scrap 1_clean.csv", encoding="utf-8-sig")
neysa2 = pd.read_csv("Neysa_Scrap 2_clean.csv", encoding="utf-8-sig")
rizky  = pd.read_csv("Rizky_Hasil scrap komentar.csv", encoding="utf-8-sig")

print(f"Neysa 1 : {len(neysa1):>5} baris")
print(f"Neysa 2 : {len(neysa2):>5} baris")
print(f"Rizky   : {len(rizky):>5} baris")

# --- Merge ---
df = pd.concat([neysa1, neysa2, rizky], ignore_index=True)
df.columns = ["text"]
df.drop_duplicates(subset="text", inplace=True)
df.dropna(subset="text", inplace=True)
df.reset_index(drop=True, inplace=True)

df.to_csv("output/data_merged.csv", index=False, encoding="utf-8")
print(f"\nSetelah merge + dedup: {len(df)} baris")

In [ ]:
# --- Fungsi Preprocessing ---
def preprocess(text: str) -> str:
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+", "", text)         # hapus URL
    text = re.sub(r"@\w+", "", text)                      # hapus @mention
    text = re.sub(r"#\w+", "", text)                      # hapus #hashtag
    text = re.sub(r"\[Sticker\]", "", text, flags=re.I)   # hapus [Sticker]
    text = re.sub(r"[^\w\s.,!?'\"()-]", " ", text)        # hapus emoji & karakter aneh
    text = re.sub(r"\s+", " ", text).strip()              # normalisasi spasi
    return text

df["text_clean"] = df["text"].apply(preprocess)
df = df[df["text_clean"].str.len() > 2].reset_index(drop=True)

df.to_csv("output/data_preprocessed.csv", index=False, encoding="utf-8")
print(f"Setelah preprocessing: {len(df)} baris")
df[["text", "text_clean"]].head(5)

---
## 🌐 Step 2 — Language Detection + Auto-Labeling

Setiap komentar dideteksi bahasanya, lalu di-route ke model transformer yang sesuai:

| Bahasa | Model |
|--------|-------|
| English (`en`) | `cardiffnlp/twitter-xlm-roberta-base-sentiment` |
| Indonesian (`id`) | `mdhugol/indonesia-bert-sentiment-classification` |
| Lainnya | `cardiffnlp/twitter-xlm-roberta-base-sentiment-multilingual` |

In [ ]:
from langdetect import detect, LangDetectException
from transformers import pipeline
import torch

df = pd.read_csv("output/data_preprocessed.csv", encoding="utf-8")
print(f"Loaded {len(df)} baris")

In [ ]:
# --- Language Detection ---
def detect_lang(text: str) -> str:
    try:
        return detect(str(text))
    except LangDetectException:
        return "en"

print("Mendeteksi bahasa...")
df["language"] = df["text_clean"].apply(detect_lang)

lang_counts = df["language"].value_counts()
print(f"Distribusi bahasa (top 10):\n{lang_counts.head(10)}")

In [ ]:
# --- Load Model Transformer ---
device = 0 if torch.cuda.is_available() else -1
print(f"Device: {'GPU' if device == 0 else 'CPU'}")

print("\nLoading model English...")
pipe_en = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    tokenizer=("cardiffnlp/twitter-xlm-roberta-base-sentiment", {"use_fast": False}),
    device=device, truncation=True, max_length=128,
)

print("Loading model Indonesian...")
pipe_id = pipeline(
    "sentiment-analysis",
    model="mdhugol/indonesia-bert-sentiment-classification",
    tokenizer=("mdhugol/indonesia-bert-sentiment-classification", {"use_fast": False}),
    device=device, truncation=True, max_length=128,
)

print("Loading model Multilingual (fallback)...")
pipe_multi = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment-multilingual",
    tokenizer=("cardiffnlp/twitter-xlm-roberta-base-sentiment-multilingual", {"use_fast": False}),
    device=device, truncation=True, max_length=128,
)

print("\nSemua model berhasil dimuat.")

In [ ]:
# --- Normalisasi Label ---
def normalize_label(raw_label: str) -> str:
    raw = raw_label.lower().strip()
    if raw in ("positive", "pos", "label_2", "positif"):
        return "Positif"
    if raw in ("negative", "neg", "label_0", "negatif"):
        return "Negatif"
    return "Netral"

# --- Routing & Inference ---
def get_sentiment(row):
    lang = row["language"]
    text = str(row["text_clean"])[:512]
    try:
        if lang == "id":
            result = pipe_id(text)[0]
        elif lang == "en":
            result = pipe_en(text)[0]
        else:
            result = pipe_multi(text)[0]
        return normalize_label(result["label"]), round(result["score"], 4)
    except Exception:
        return "Netral", 0.0

print("Menjalankan auto-labeling (beberapa menit)...")
results = df.apply(get_sentiment, axis=1)
df["label"]      = [r[0] for r in results]
df["confidence"] = [r[1] for r in results]

df.to_csv("output/data_labeled.csv", index=False, encoding="utf-8")
print(f"\nSelesai! Distribusi label:")
print(df["label"].value_counts())
df[["text_clean", "language", "label", "confidence"]].head(10)

---
## 🤖 Step 3 — Training SVM & Evaluasi

Label dari transformer dipakai sebagai **ground truth**. SVM dilatih pada 80% data, lalu dievaluasi pada 20% sisanya.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

os.makedirs("output/figures", exist_ok=True)

df = pd.read_csv("output/data_labeled.csv", encoding="utf-8")
df = df.dropna(subset=["text_clean", "label"])
print(f"Data: {len(df)} baris")
print(df["label"].value_counts())

In [ ]:
X = df["text_clean"].astype(str)
y = df["label"]

# 80/20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# TF-IDF: unigram + bigram, max 10.000 fitur
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), sublinear_tf=True)
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec  = tfidf.transform(X_test)

# LinearSVC
svm = LinearSVC(max_iter=2000)
svm.fit(X_train_vec, y_train)
y_pred_svm = svm.predict(X_test_vec)

svm_acc = accuracy_score(y_test, y_pred_svm)
print(f"\n=== SVM Accuracy: {svm_acc:.4f} ({svm_acc*100:.1f}%) ===")
print()
print(classification_report(y_test, y_pred_svm))

In [ ]:
# --- Confusion Matrix ---
labels = ["Positif", "Negatif", "Netral"]
cm = confusion_matrix(y_test, y_pred_svm, labels=labels)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_title("Confusion Matrix — SVM", fontsize=13)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual (Transformer Label)")
plt.tight_layout()
plt.savefig("output/figures/confusion_matrix_svm.png", dpi=150)
plt.show()

In [ ]:
# --- Perbandingan SVM Accuracy vs Transformer Avg Confidence ---
avg_transformer_conf = df["confidence"].mean()

fig, ax = plt.subplots(figsize=(6, 4))
models = ["SVM\n(vs Transformer label)", "Transformer\n(avg confidence)"]
scores = [svm_acc, avg_transformer_conf]
colors = ["#4C72B0", "#DD8452"]
bars = ax.bar(models, scores, color=colors, width=0.4)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("SVM Accuracy vs Transformer Avg Confidence")
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.02, f"{score:.3f}", ha="center", fontsize=11)
plt.tight_layout()
plt.savefig("output/figures/model_comparison.png", dpi=150)
plt.show()

# Simpan report
with open("output/svm_report.txt", "w", encoding="utf-8") as f:
    f.write(f"SVM Accuracy: {svm_acc:.4f}\n\n")
    f.write(classification_report(y_test, y_pred_svm))
    f.write(f"\nTransformer avg confidence: {avg_transformer_conf:.4f}\n")
print("Report tersimpan di output/svm_report.txt")

---
## 📊 Step 4 — Visualisasi Hasil

In [ ]:
from wordcloud import WordCloud

df = pd.read_csv("output/data_labeled.csv", encoding="utf-8")
df = df.dropna(subset=["text_clean", "label"])

COLORS = {"Positif": "#2ecc71", "Negatif": "#e74c3c", "Netral": "#3498db"}
label_counts = df["label"].value_counts()
print(f"Total data: {len(df)} komentar")
print(label_counts)

In [ ]:
# --- Pie Chart & Bar Chart ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Pie
ax1.pie(
    label_counts,
    labels=label_counts.index,
    autopct="%1.1f%%",
    colors=[COLORS.get(l, "#aaa") for l in label_counts.index],
    startangle=140,
    wedgeprops={"edgecolor": "white", "linewidth": 1.5},
)
ax1.set_title("Distribusi Sentimen Komentar TikTok\n(Tema: AI Menggantikan Manusia)", fontsize=12)

# Bar
bars = ax2.bar(label_counts.index, label_counts.values,
               color=[COLORS.get(l, "#aaa") for l in label_counts.index],
               edgecolor="white")
for bar, val in zip(bars, label_counts.values):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
             str(val), ha="center", fontsize=11)
ax2.set_xlabel("Sentimen")
ax2.set_ylabel("Jumlah Komentar")
ax2.set_title("Distribusi Sentimen Komentar TikTok", fontsize=12)

plt.tight_layout()
plt.savefig("output/figures/sentiment_distribution_combined.png", dpi=150)
plt.show()

In [ ]:
# --- WordCloud per Sentimen ---
cmap_map = {"Positif": "Greens", "Negatif": "Reds", "Netral": "Blues"}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, label in zip(axes, ["Positif", "Negatif", "Netral"]):
    subset = df[df["label"] == label]["text_clean"].dropna()
    if subset.empty:
        ax.axis("off")
        continue
    combined = " ".join(subset.tolist())
    wc = WordCloud(
        width=600, height=350,
        background_color="white",
        colormap=cmap_map[label],
        max_words=100,
        collocations=False,
    ).generate(combined)
    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    ax.set_title(f"WordCloud — {label}", fontsize=13)

plt.tight_layout()
plt.savefig("output/figures/wordcloud_all.png", dpi=150)
plt.show()

In [ ]:
# --- Distribusi Sentimen per Bahasa (Top 6) ---
top_langs = df["language"].value_counts().head(6).index
lang_label = df.groupby(["language", "label"]).size().unstack(fill_value=0)
lang_label = lang_label.loc[lang_label.index.isin(top_langs)]

fig, ax = plt.subplots(figsize=(9, 5))
lang_label.plot(kind="bar", ax=ax,
                color=[COLORS.get(c, "#aaa") for c in lang_label.columns],
                edgecolor="white")
ax.set_xlabel("Bahasa")
ax.set_ylabel("Jumlah Komentar")
ax.set_title("Distribusi Sentimen per Bahasa (Top 6)", fontsize=13)
ax.legend(title="Sentimen")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("output/figures/sentiment_per_language.png", dpi=150)
plt.show()

print("\n✅ Semua visualisasi selesai. File tersimpan di output/figures/")